In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

# Loading all datasets
train = pd.read_csv('../data/train.csv', parse_dates=['date'])
stores = pd.read_csv('../data/stores.csv')
oil = pd.read_csv('../data/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('../data/holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('../data/transactions.csv', parse_dates=['date'])

print(f"Train shape: {train.shape}")
print("Data loaded. Ready for feature engineering.")

Train shape: (3000888, 6)
Data loaded. Ready for feature engineering.


In [4]:
# Basic calendar features
train['day_of_week'] = train['date'].dt.dayofweek        # 0=Monday, 6=Sunday
train['day_of_month'] = train['date'].dt.day
train['month'] = train['date'].dt.month
train['year'] = train['date'].dt.year
train['week_of_year'] = train['date'].dt.isocalendar().week.astype(int)

# Is it a weekend? (Saturday=5, Sunday=6)
train['is_weekend'] = (train['day_of_week'] >= 5).astype(int)

# Is it payday? In Ecuador, people typically get paid on the 15th and last day of month
train['is_payday'] = (
    (train['day_of_month'] == 15) | 
    (train['date'].dt.is_month_end)
).astype(int)

# Is it beginning of month? (people spend more right after payday)
train['is_month_start'] = (train['day_of_month'] <= 3).astype(int)

print("Time features added:")
print(train[['date', 'day_of_week', 'month', 'year', 'is_weekend', 'is_payday', 'is_month_start']].head(40))
print(f"\nShape: {train.shape}")

Time features added:
         date  day_of_week  month  year  is_weekend  is_payday  is_month_start
0  2013-01-01            1      1  2013           0          0               1
1  2013-01-01            1      1  2013           0          0               1
2  2013-01-01            1      1  2013           0          0               1
3  2013-01-01            1      1  2013           0          0               1
4  2013-01-01            1      1  2013           0          0               1
5  2013-01-01            1      1  2013           0          0               1
6  2013-01-01            1      1  2013           0          0               1
7  2013-01-01            1      1  2013           0          0               1
8  2013-01-01            1      1  2013           0          0               1
9  2013-01-01            1      1  2013           0          0               1
10 2013-01-01            1      1  2013           0          0               1
11 2013-01-01            1     

all the rows show the same date because each date has 33 product families × 54 stores. That's expected.


In [5]:
train = train.merge(stores, on='store_nbr', how='left')

print("Store features merged:")
print(train[['date', 'store_nbr', 'family', 'city', 'state', 'type', 'cluster']].head())
print(f"\nStore types: {train['type'].unique()}")
print(f"Clusters:    {sorted(train['cluster'].unique())}")
print(f"\nShape: {train.shape}")

Store features merged:
        date  store_nbr      family   city      state type  cluster
0 2013-01-01          1  AUTOMOTIVE  Quito  Pichincha    D       13
1 2013-01-01          1   BABY CARE  Quito  Pichincha    D       13
2 2013-01-01          1      BEAUTY  Quito  Pichincha    D       13
3 2013-01-01          1   BEVERAGES  Quito  Pichincha    D       13
4 2013-01-01          1       BOOKS  Quito  Pichincha    D       13

Store types: <StringArray>
['D', 'C', 'B', 'E', 'A']
Length: 5, dtype: str
Clusters:    [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17)]

Shape: (3000888, 18)


In [6]:
oil['dcoilwtico'] = oil['dcoilwtico'].ffill()

# Backfill the very first row if it's NaN
oil['dcoilwtico'] = oil['dcoilwtico'].bfill()

train = train.merge(oil, on='date', how='left')

# Forward fill any remaining NaN oil prices in train
train['dcoilwtico'] = train['dcoilwtico'].ffill()
train['dcoilwtico'] = train['dcoilwtico'].bfill()

print("Oil prices merged:")
print(train[['date', 'store_nbr', 'family', 'sales', 'dcoilwtico']].head())
print(f"\nOil price NaN count: {train['dcoilwtico'].isna().sum()}")
print(f"\nShape: {train.shape}")

Oil prices merged:
        date  store_nbr      family  sales  dcoilwtico
0 2013-01-01          1  AUTOMOTIVE    0.0       93.14
1 2013-01-01          1   BABY CARE    0.0       93.14
2 2013-01-01          1      BEAUTY    0.0       93.14
3 2013-01-01          1   BEVERAGES    0.0       93.14
4 2013-01-01          1       BOOKS    0.0       93.14

Oil price NaN count: 0

Shape: (3000888, 19)


Holiday feature



In [7]:
# Focusing on national holidays first (affect all stores)
national_holidays = holidays[
    (holidays['locale'] == 'National') & 
    (holidays['transferred'] == False)
]['date'].unique()

# Transferring holidays (moved to another day)
transferred = holidays[holidays['type'] == 'Transfer']['date'].unique()

# Creating  holiday flags
train['is_national_holiday'] = train['date'].isin(national_holidays).astype(int)
train['is_transferred_holiday'] = train['date'].isin(transferred).astype(int)

# Regional holidays - match by state
regional_holidays = holidays[holidays['locale'] == 'Regional'][['date', 'locale_name']]
train = train.merge(
    regional_holidays.rename(columns={'locale_name': 'state_match'}),
    left_on=['date', 'state'],
    right_on=['date', 'state_match'],
    how='left'
)
train['is_regional_holiday'] = train['state_match'].notna().astype(int)
train.drop(columns=['state_match'], inplace=True)

# Combined holiday flag
train['is_holiday'] = (
    (train['is_national_holiday'] == 1) | 
    (train['is_regional_holiday'] == 1) | 
    (train['is_transferred_holiday'] == 1)
).astype(int)

print("Holiday features added:")
print(f"National holiday days:    {train['is_national_holiday'].sum() // (54*33)}")
print(f"Regional holiday entries: {train['is_regional_holiday'].sum()}")
print(f"Transferred holidays:    {train['is_transferred_holiday'].sum() // (54*33)}")

Holiday features added:
National holiday days:    136
Regional holiday entries: 1023
Transferred holidays:    9


Lag features and rolling avergaes :

In [10]:
#Sort by store, family, and date (critical for correct lag calculation)
train = train.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

# Group by store-family combination for lag calculations
# Each group is one product family in one store over time
train['lag_7'] = train.groupby(['store_nbr', 'family'])['sales'].shift(7)    # Last week same day
train['lag_14'] = train.groupby(['store_nbr', 'family'])['sales'].shift(14)  # Two weeks ago
train['lag_28'] = train.groupby(['store_nbr', 'family'])['sales'].shift(28)  # Four weeks ago

# Rolling averages (smooth out daily noise)
train['rolling_7_mean'] = train.groupby(['store_nbr', 'family'])['sales'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).mean()
)
train['rolling_14_mean'] = train.groupby(['store_nbr', 'family'])['sales'].transform(
    lambda x: x.shift(1).rolling(window=14, min_periods=1).mean()
)
train['rolling_28_mean'] = train.groupby(['store_nbr', 'family'])['sales'].transform(
    lambda x: x.shift(1).rolling(window=28, min_periods=1).mean()
)

print("Lag and rolling features added:")
print(train[['date', 'store_nbr', 'family', 'sales', 'lag_7', 'rolling_7_mean', 'rolling_28_mean']].dropna().head(10))
print(f"\nShape: {train.shape}")
print(f"NaN counts in lag features:")
print(f"  lag_7:  {train['lag_7'].isna().sum():,}")
print(f"  lag_14: {train['lag_14'].isna().sum():,}")
print(f"  lag_28: {train['lag_28'].isna().sum():,}")

Lag and rolling features added:
         date  store_nbr      family  sales  lag_7  rolling_7_mean  \
7  2013-01-08          1  AUTOMOTIVE    2.0    0.0        2.142857   
8  2013-01-09          1  AUTOMOTIVE    2.0    2.0        2.428571   
9  2013-01-10          1  AUTOMOTIVE    2.0    3.0        2.428571   
10 2013-01-11          1  AUTOMOTIVE    3.0    3.0        2.285714   
11 2013-01-12          1  AUTOMOTIVE    2.0    5.0        2.285714   
12 2013-01-13          1  AUTOMOTIVE    2.0    2.0        1.857143   
13 2013-01-14          1  AUTOMOTIVE    2.0    0.0        1.857143   
14 2013-01-15          1  AUTOMOTIVE    1.0    2.0        2.142857   
15 2013-01-16          1  AUTOMOTIVE    1.0    2.0        2.000000   
16 2013-01-17          1  AUTOMOTIVE    1.0    2.0        1.857143   

    rolling_28_mean  
7          2.142857  
8          2.125000  
9          2.111111  
10         2.100000  
11         2.181818  
12         2.166667  
13         2.153846  
14         2.142857  

Encode categorical variaables our model cannot read word.

In [11]:
from sklearn.preprocessing import LabelEncoder

# Columns to encode
cat_columns = ['family', 'city', 'state', 'type']

label_encoders = {}
for col in cat_columns:
    le = LabelEncoder()
    train[f'{col}_encoded'] = le.fit_transform(train[col])
    label_encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique values encoded")

print(f"\nShape: {train.shape}")
print(f"\nSample encoded values:")
print(train[['family', 'family_encoded', 'type', 'type_encoded', 'city', 'city_encoded']].drop_duplicates().head(10))

family: 33 unique values encoded
city: 22 unique values encoded
state: 16 unique values encoded
type: 5 unique values encoded

Shape: (3000888, 33)

Sample encoded values:
             family  family_encoded type  type_encoded   city  city_encoded
0        AUTOMOTIVE               0    D             3  Quito            18
1684      BABY CARE               1    D             3  Quito            18
3368         BEAUTY               2    D             3  Quito            18
5052      BEVERAGES               3    D             3  Quito            18
6736          BOOKS               4    D             3  Quito            18
8420   BREAD/BAKERY               5    D             3  Quito            18
10104   CELEBRATION               6    D             3  Quito            18
11788      CLEANING               7    D             3  Quito            18
13472         DAIRY               8    D             3  Quito            18
15156          DELI               9    D             3  Quito       

Preparing final data set.

In [12]:
feature_columns = [
    # Time features
    'day_of_week', 'day_of_month', 'month', 'year', 'week_of_year',
    'is_weekend', 'is_payday', 'is_month_start',
    
    # Store features
    'store_nbr', 'cluster', 'type_encoded', 'city_encoded', 'state_encoded',
    
    # Product feature
    'family_encoded',
    
    # Promotion
    'onpromotion',
    
    # Oil price
    'dcoilwtico',
    
    # Holiday features
    'is_national_holiday', 'is_regional_holiday', 'is_transferred_holiday', 'is_holiday',
    
    # Lag features
    'lag_7', 'lag_14', 'lag_28',
    
    # Rolling averages
    'rolling_7_mean', 'rolling_14_mean', 'rolling_28_mean'
]

target = 'sales'

# Drop rows with NaN in lag features (first 28 days per store-family)
df_model = train.dropna(subset=['lag_28']).reset_index(drop=True)

print(f"Original train size:     {len(train):,}")
print(f"After dropping NaN lags: {len(df_model):,}")
print(f"Rows dropped:           {len(train) - len(df_model):,}")
print(f"\nFeature count: {len(feature_columns)}")
print(f"\nFeatures:\n{feature_columns}")
print(f"\nTarget: {target}")
print(f"\nAny remaining NaN in features: {df_model[feature_columns].isna().any().any()}")

# Save the processed dataset
df_model.to_csv('../data/train_processed.csv', index=False)
print(f"\nProcessed dataset saved to data/train_processed.csv")

Original train size:     3,000,888
After dropping NaN lags: 2,950,992
Rows dropped:           49,896

Feature count: 26

Features:
['day_of_week', 'day_of_month', 'month', 'year', 'week_of_year', 'is_weekend', 'is_payday', 'is_month_start', 'store_nbr', 'cluster', 'type_encoded', 'city_encoded', 'state_encoded', 'family_encoded', 'onpromotion', 'dcoilwtico', 'is_national_holiday', 'is_regional_holiday', 'is_transferred_holiday', 'is_holiday', 'lag_7', 'lag_14', 'lag_28', 'rolling_7_mean', 'rolling_14_mean', 'rolling_28_mean']

Target: sales

Any remaining NaN in features: False

Processed dataset saved to data/train_processed.csv
